> **Data note:** this notebook was run on a private clinical cohort that is not included in this repository, so it will not run as-is. The public diabetes dataset used in the paper is in `data/`; see `03_rail_d.ipynb` and `04_c_rail_d.ipynb` for runnable versions.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

from dotenv import load_dotenv, find_dotenv
import requests
import os

from pydantic import BaseModel, Field
from typing import Literal
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import ChatPromptTemplate

from tqdm import tqdm

from langchain_openai import ChatOpenAI
import json

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score

In [ ]:
@dataclass
class KNNState:
    df: pd.DataFrame
    feature_cols: List[str]
    medians: pd.Series
    scaler: StandardScaler
    feats_imputed: np.ndarray   # NEW: features after numeric coercion + median impute
    X_scaled: np.ndarray        # normalized features used for KNN
    nn: NearestNeighbors
    excel_offset: int

In [ ]:

def build_state(
    df: pd.DataFrame,
    use_cols: Optional[List[str]] = None,
    excel_header_on_row1: bool = True,
) -> KNNState:
    # 1) Choose feature columns (first 24 = A..X by position unless `use_cols` provided)
    feature_cols = use_cols if use_cols is not None else list(df.columns[:24])

    # 2) Coerce to numeric + median impute
    feats = df[feature_cols].apply(pd.to_numeric, errors="coerce")
    medians = feats.median(numeric_only=True)
    feats = feats.fillna(medians)
    feats_imputed = feats.values  # keep for printing

    # 3) Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(feats_imputed)

    # 4) Fit NN (Euclidean in normalized space)
    nn = NearestNeighbors(metric="euclidean")
    nn.fit(X_scaled)

    excel_offset = 2 if excel_header_on_row1 else 1
    return KNNState(
        df=df,
        feature_cols=feature_cols,
        medians=medians,
        scaler=scaler,
        feats_imputed=feats_imputed,
        X_scaled=X_scaled,
        nn=nn,
        excel_offset=excel_offset,
    )

In [ ]:
#find the closest neighbors

def closest_indices_for_row(
    row_idx: int,
    state: KNNState,
    k: int = 5,
    return_distances: bool = False,
) -> np.ndarray | Tuple[np.ndarray, np.ndarray]:
    """
    Given a row index from the same DataFrame used in `build_state`,
    return indices of the k closest rows (excluding the row itself).
    Optionally return distances as well.

    Returns:
      - indices: shape (k,) 0-based row indices
      - distances (optional): shape (k,) Euclidean distances in normalized space
    """
    # Query for k+1 (self + k neighbors), then drop self
    distances, indices = state.nn.kneighbors(
        state.X_scaled[row_idx].reshape(1, -1),
        n_neighbors=k + 1
    )
    inds = indices[0].tolist()
    dists = distances[0].tolist()

    # Remove self if present
    if row_idx in inds:
        j = inds.index(row_idx)
        inds.pop(j)
        dists.pop(j)

    # Trim in case self wasn't present for some reason
    inds = np.array(inds[:k], dtype=int)
    dists = np.array(dists[:k], dtype=float)

    return (inds, dists) if return_distances else inds

In [ ]:
def pretty_print_row(row_idx: int, state: KNNState):
    """Prints the full Excel row (all columns) for the given row index."""
    row_df = pd.DataFrame([state.df.iloc[row_idx]], columns=state.df.columns)
    print(row_df.to_string(index=False))

In [ ]:
#format the few shot examples

def format_fewshot(row_idx: int, state: KNNState) -> dict:
    """
    Return a row as a few-shot example object:
    - features: dict of columns A..X
    - label: value from column Y
    - rationale: dict with AB, AC, AD
    """
    # Identify relevant columns
    feature_cols = list(state.df.columns[:24])      # A..X
    label_col = state.df.columns[24]               # Y
    reasoning_cols = list(state.df.columns[27:30]) # AB, AC, AD

    row = state.df.iloc[row_idx]

    return {
        "patient_id": row_idx,
        "features": row[feature_cols].to_dict(),
        #"rationale": {col: row[col] for col in reasoning_cols},
        "label": row[label_col]
    }

In [ ]:
def render_few_shots_block(df: pd.DataFrame, neighbor_indices: list[int], state: KNNState) -> str:
    """
    Build a text block for the human prompt with multiple few-shot examples.
    Each example has: features JSON (A..X), IEC_HS label, and reasoning (AB/AC/AD).
    """
    lines = []
    for i, idx in enumerate(neighbor_indices, start=1):
        ex = format_fewshot(idx, state)
        lines.append(
        f"patient_id: {ex['patient_id']}\n"
        f"features_json: {ex['features']}\n"
        #f"rationale: {ex['rationale']}\n"
        f"label: {ex['label']}\n"
    )
    return "\n".join(lines).strip()

In [ ]:
#all configurations

#EXCEL_PATH = "../data/Patients_toy.xlsx"       # adjust path if neede
EXCEL_PATH = "../data/iec_hs_rationales.xlsx"
SHEET = 0                                           # or a sheet name
MODEL = "gpt-4o-mini"   # any OpenAI chat model
TEMPERATURE = 0.0                                   # deterministic classification
TARGET_COL = "IEC_HS"                               # ground-truth label in your file
OUT_CSV = "../outputs/granite3-1-8B-RAIL.csv"

In [ ]:
# OpenAI API access: put OPENAI_API_KEY in your environment or in a local .env file
# (see .env.example). Never commit your key.
load_dotenv(find_dotenv())
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set")

In [ ]:
# strcutured output and getting the prompt

class IecHsPrediction(BaseModel):
    rationale: str = Field(
        ..., description="key supporting evidence for IEC_HS to be positive as well as contradicting factors. Followed by overall reasoning for what should be the prediction for IEC_HS."
    )
    iec_hs: Literal[0, 1] = Field(..., description="Predicted IEC_HS class label (0 or 1).")
    confidence: float = Field(
        ..., ge=0.0, le=1.0,
        description="Model self-reported confidence between 0 and 1."
    )
   

#parser = JsonOutputParser(pydantic_object=IecHsPrediction)
parser = PydanticOutputParser(pydantic_object=IecHsPrediction)

system_msg = """You are a careful clinical classifier. 
Given a single patient's anonymized features, predict whether IEC_HS = 0 or 1.
Only use the provided features—do not invent data.

Output MUST be valid JSON matching this schema:
- rationale: short reasoning for prediction (3-5 sentences)
- iec_hs: 0 or 1
- confidence: a number in [0, 1]
"""

human_template = """
### Few-shot examples
{few_shots_block}

### Patient to classify
Patient features (key → value) as JSON:
{patient_json}

Instructions:
1) Consider all the supporting as well as contradicting evidences and reason about what the prediction of IEC_HS should be.
2) Predict IEC_HS (0 or 1).
3) Return ONLY JSON, no extra text.

Use this format:
{format_instructions}

Output MUST be valid JSON matching this schema:
- rationale: short reasoning for prediction (3-5 sentences)
- iec_hs: 0 or 1
- confidence: a number in [0, 1]
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_msg), ("human", human_template)]
)

# IMPORTANT: inject the raw JSON schema as a *variable*, not as literal text
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

In [ ]:
llm = ChatOpenAI(
        model=MODEL,
        temperature=0,
        max_retries=2,
        max_tokens=2048,
        api_key=api_key,
    )

In [ ]:
chain = prompt | llm | parser

In [ ]:
df = pd.read_excel("../data/iec_hs_rationales.xlsx")
selected_cols = ["CRP", "Diagnosis", "Ferritin", "Tocilizumab given"]
state = build_state(df, use_cols=selected_cols)
#state = build_state(df)  # uses first 24 columns (A..X) by default
test_row = 11
nn_inds, nn_dists = closest_indices_for_row(test_row, state, k=4, return_distances=True)
print(render_few_shots_block(df, nn_inds, state))

In [ ]:
results = []
errored_rows = []  # to keep track of rows where exceptions happened

#selected_cols = ["CRP", "Diagnosis", "Ferritin", "Tocilizumab given"]

#state = build_state(df, use_cols=selected_cols)

state = build_state(df)  # uses first 24 columns (A..X) by default

random_index = 6
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Scoring patients"):
    try: 
        nn_inds, nn_dists = closest_indices_for_row(random_index, state, k=6, return_distances=True)
        few_shot_block = render_few_shots_block(df, nn_inds, state)
        record = format_fewshot(idx, state)
        patient_features = record["features"]
        # Safety: ensure we never leak the ground-truth label to the model
        assert TARGET_COL not in patient_features
        #print(f"patient features are {patient_features}")
        #print(f"few shot are {few_shot_block}")

        pred: IecHsPrediction = chain.invoke({
            "few_shots_block": few_shot_block,
            "patient_json": json.dumps(patient_features, ensure_ascii=False)
        })
        #print(f"prediction is {pred['iec_hs']}")
        print(f"prediction is {pred.iec_hs} with confidence {pred.confidence}  and true label {row.get(TARGET_COL, None)}")
        results.append({
                "row_index": idx,
                "iec_hs_pred": int(pred.iec_hs),
                "confidence": float(pred.confidence),
                "rationale": pred.rationale,
                "iec_hs_true": row.get(TARGET_COL, None)
            })
    except Exception as e:
        errored_rows.append((idx, str(e)))  # store row index + error message
        print(f"prediction is SKIPPED {e}")
        continue  # skip to next row

In [ ]:
out_df = pd.DataFrame(results)

# Keep only rows where both are non-null
valid = out_df["iec_hs_pred"].notna() & out_df["iec_hs_true"].notna()
y_true = out_df.loc[valid, "iec_hs_true"].astype(int)
y_pred = out_df.loc[valid, "iec_hs_pred"].astype(int)

# Overall metrics (positive class = 1)
accuracy   = accuracy_score(y_true, y_pred)
precision1 = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
recall1    = recall_score(y_true, y_pred,    pos_label=1, zero_division=0)
f1_pos     = f1_score(y_true, y_pred,        pos_label=1, zero_division=0)

# Negative-class metrics (label = 0)
precision0 = precision_score(y_true, y_pred, pos_label=0, zero_division=0)
recall0    = recall_score(y_true, y_pred,    pos_label=0, zero_division=0)
f1_neg     = f1_score(y_true, y_pred,        pos_label=0, zero_division=0)

# Macro F1 (unweighted mean of per-class F1s)
f1_macro   = f1_score(y_true, y_pred, average="macro", zero_division=0)

print(f"Accuracy:     {accuracy:.3f}")
print(f"Pos P/R/F1:   P={precision1:.3f}  R={recall1:.3f}  F1={f1_pos:.3f}")
print(f"Neg P/R/F1:   P={precision0:.3f}  R={recall0:.3f}  F1={f1_neg:.3f}")
print(f"Macro F1:     {f1_macro:.3f}")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
print("\nConfusion Matrix (rows = true, cols = pred):")
print("         Pred 0   Pred 1")
print(f"True 0   {cm[0,0]:6}   {cm[0,1]:6}")
print(f"True 1   {cm[1,0]:6}   {cm[1,1]:6}")

In [ ]:
# Save results
out_df = pd.DataFrame(results)
print(out_df.columns.tolist())
# Optional: simple accuracy if labels exist
if TARGET_COL in df.columns and out_df["iec_hs_pred"].notna().any():

    acc = (out_df["iec_hs_pred"] == out_df["iec_hs_true"]).mean()
    print(f"Accuracy on labeled rows: {acc:.3f}")

#print(out_df)
out_df.to_csv(OUT_CSV, index=False)